# Week 5 — Functions & Modules
### Python for Blockchain Analytics | Phase 1

---

**What you'll learn this week:**
- Defining functions — `def`, parameters, return values
- Default arguments and keyword arguments
- `*args` and `**kwargs` — flexible function signatures
- Lambda functions — short anonymous functions
- Scope — local vs global variables
- Modules — importing and writing your own
- Building `blockchain_utils.py` — your personal toolkit

**SQL analyst parallel:**
- A function ≈ a saved query or a stored procedure
- A module ≈ a schema full of stored procedures you can reuse
- Default arguments ≈ optional WHERE clause parameters with defaults
- `*args` ≈ a variadic function that accepts any number of inputs

**Time:** ~3 hours

---

## 1. Defining Functions

A function is a named, reusable block of code.
You define it once and call it as many times as you need.

**Why functions matter:**
- DRY — Don't Repeat Yourself
- Testable — you can test one function in isolation
- Readable — a good function name explains what the code does

In [1]:
# Basic function — no parameters, no return value
def print_separator():
    print("=" * 50)

print_separator()
print("ETH Price: $3,247.85")
print_separator()

# Function with parameters
def format_address(address):
    """Shorten a wallet address to 0xABCD...1234 format."""
    if len(address) < 10:
        return address
    return f"{address[:6]}...{address[-4:]}"

addr = "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045"
print(format_address(addr))

# Function with return value
def wei_to_eth(wei):
    """Convert Wei to ETH."""
    return wei / 10**18

balance_wei = 1_500_000_000_000_000_000
print(f"Balance: {wei_to_eth(balance_wei):.4f} ETH")

ETH Price: $3,247.85
0xd8dA...6045
Balance: 1.5000 ETH


In [2]:
# Multiple parameters
def format_token_amount(amount, symbol, decimals=18):
    """Convert raw token amount to human-readable with symbol."""
    human = amount / 10**decimals
    return f"{human:,.6f} {symbol}"

# ETH (18 decimals)
print(format_token_amount(1_500_000_000_000_000_000, "ETH"))

# USDC (6 decimals)
print(format_token_amount(1_500_000, "USDC", decimals=6))

# Returning multiple values as a tuple
def price_stats(prices):
    """Return (min, max, average) for a list of prices."""
    return min(prices), max(prices), sum(prices) / len(prices)

eth_prices = [3100.0, 3247.85, 3310.0, 3290.0, 3400.0]
low, high, avg = price_stats(eth_prices)
print(f"Range: ${low:,.2f} – ${high:,.2f} | Avg: ${avg:,.2f}")

1.500000 ETH
1.500000 USDC
Range: $3,100.00 – $3,400.00 | Avg: $3,269.57


## 2. Default Arguments & Keyword Arguments

Default arguments give parameters a fallback value —
callers don't need to pass them unless they want to override.

In [3]:
# Default arguments — must come AFTER required arguments
def get_wallet_summary(address, include_failed=False, currency="USD", decimals=4):
    """
    Summarise a wallet's activity.
    address      — required
    include_failed — optional, default False
    currency     — optional, default USD
    decimals     — optional, default 4
    """
    # Simulated data
    data = {
        "address":  address,
        "eth_balance": 2.5478,
        "tx_count": 142,
        "failed_tx": 8,
        "total_volume_eth": 847.32,
    }

    label = f"Wallet: {address[:10]}..."
    balance = f"{data['eth_balance']:.{decimals}f} ETH"
    txns    = data["tx_count"]
    if include_failed:
        txns = f"{txns} (+ {data['failed_tx']} failed)"
    return f"{label} | Balance: {balance} | Txns: {txns}"

# Call with only required argument
print(get_wallet_summary("0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045"))

# Override some defaults
print(get_wallet_summary("0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
                          include_failed=True))

# Keyword arguments — name them explicitly (order doesn't matter)
print(get_wallet_summary(
    "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
    decimals=2,
    include_failed=True,
    currency="ETH",
))

Wallet: 0xd8dA6BF2... | Balance: 2.5478 ETH | Txns: 142
Wallet: 0xd8dA6BF2... | Balance: 2.5478 ETH | Txns: 142 (+ 8 failed)
Wallet: 0xd8dA6BF2... | Balance: 2.55 ETH | Txns: 142 (+ 8 failed)


In [4]:
# ⚠️ The mutable default argument trap 

# This is WRONG declaration — list as default is shared across all calls
def add_to_watchlist_BAD(address, watchlist=[]):
    watchlist.append(address)
    return watchlist

print(add_to_watchlist_BAD("0xAlice"))   # ['0xAlice']
print(add_to_watchlist_BAD("0xBob"))     # ['0xAlice', '0xBob'] ← BUG! Alice is still there

# CORRECT — use None as default, create inside function
def add_to_watchlist(address, watchlist=None):
    if watchlist is None:
        watchlist = []
    watchlist.append(address)
    return watchlist

print(add_to_watchlist("0xAlice"))  # ['0xAlice']
print(add_to_watchlist("0xBob"))    # ['0xBob'] ← correct, fresh list each time

print("""
Rule: never use mutable objects (list, dict, set) as default arguments.
Use None instead and create the object inside the function.
""")

['0xAlice']
['0xAlice', '0xBob']
['0xAlice']
['0xBob']

Rule: never use mutable objects (list, dict, set) as default arguments.
Use None instead and create the object inside the function.



## 3. *args and **kwargs

`*args` — Variable Position arguments accept any number of positional arguments (collected as a tuple)
    
 `**kwargs` — Variable Keyword arguments accept any number of keyword arguments (collected as a dict)

**When you need them:**
- Building flexible API wrapper functions
- Writing utilities that work on multiple tokens/addresses at once
- Passing arbitrary config options through to another function

In [5]:
# *args — variable positional arguments
def total_portfolio_value(*token_values):
    """
    Calculate total portfolio value from any number of token USD values.
    *token_values collects all positional args into a tuple.
    """
    return sum(token_values)

# Call with any number of arguments
print(f"2-token portfolio: ${total_portfolio_value(12_179.44, 8_089.44):,.2f}")
print(f"4-token portfolio: ${total_portfolio_value(12_179.44, 8_089.44, 4_250.00, 642.00):,.2f}")

# *args in practice — batch address formatter
def format_addresses(*addresses):
    """Format any number of addresses for display."""
    return [f"{a[:6]}...{a[-4:]}" for a in addresses]

result = format_addresses(
    "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
    "0x1f9840a85d5aF5bf1D1762F925BDADdC4201F984",
    "0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D",
)
for r in result:
    print(f"  {r}")

2-token portfolio: $20,268.88
4-token portfolio: $25,160.88
  0xd8dA...6045
  0x1f98...F984
  0x7a25...488D


In [8]:
# **kwargs — variable keyword arguments
def build_api_request(endpoint, **params):
    """
    Build a query string for a blockchain API.
    **params collects all keyword args into a dict.
    """
    base_url = f"https://api.etherscan.io/{endpoint}"
    if not params:
        return base_url
    query = "&".join(f"{k}={v}" for k, v in params.items())
    return f"{base_url}?{query}"

# Flexible — pass any params you need
print(build_api_request("api", module="account", action="txlist",
                         address="0xd8dA...", startblock=0, endblock=99999999, blockchain='base'))

https://api.etherscan.io/api?module=account&action=txlist&address=0xd8dA...&startblock=0&endblock=99999999&blockchain=base


In [17]:
# Combining *args and **kwargs in a funtion
def log_event(event_type, *addresses, **metadata):
    """
    Log a blockchain event with flexible arguments.
    event_type — required
    *addresses — any number of involved addresses
    **metadata — any additional context
    """
    print(f"...[{event_type.upper()}]...")
    for i, addr in enumerate(addresses):
        print(f"  Address {i+1}: {addr[:10]}...")
    for key, val in metadata.items():
        print(f"  {key}: {val}")

log_event(
    "swap",
    "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
    "0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D",
    block=19_847_293,
    token_in="USDC",
    token_out="ETH",
    amount_in=1000,
)

...[SWAP]...
  Address 1: 0xd8dA6BF2...
  Address 2: 0x7a250d56...
  block: 19847293
  token_in: USDC
  token_out: ETH
  amount_in: 1000


## 4. Lambda Functions

A lambda is a small, anonymous, one-line function.
Best used as an argument to `sorted()`, `map()`, `filter()`.

**Rule:** if your lambda needs more than one line, write a proper `def` instead.

In [18]:
# Lambda syntax: lambda arguments: expression
double = lambda x: x * 2
print(double(5))   # 10

# Most common use: as key= argument in sorted()
tokens = [
    {"symbol": "UNI",  "price": 12.84,  "volume": 180_000_000},
    {"symbol": "ETH",  "price": 3247.85,"volume": 15_800_000_000},
    {"symbol": "AAVE", "price": 98.50,  "volume": 95_000_000},
    {"symbol": "ARB",  "price": 1.24,   "volume": 420_000_000},
]

# Sort by price
by_price  = sorted(tokens, key=lambda t: t["price"])
print("Cheapest first:", [t["symbol"] for t in by_price])

# Sort by volume descending
by_volume = sorted(tokens, key=lambda t: -t["volume"])
print("Most volume:   ", [t["symbol"] for t in by_volume])

# Sort by market cap proxy (price × arbitrary supply — just for demo)
by_mcap = sorted(tokens, key=lambda t: t["price"] * t["volume"], reverse=True)
print("By price×vol:  ", [t["symbol"] for t in by_mcap])

# Lambda with map() — apply a function to every item in a list
wei_amounts = [1_500_000_000_000_000_000, 500_000_000_000_000_000, 250_000_000_000_000_000]
eth_amounts = list(map(lambda w: w / 10**18, wei_amounts))
print("ETH amounts:", eth_amounts)

# Lambda with filter() — keep only items matching a condition
large_tokens = list(filter(lambda t: t["price"] > 50, tokens))
print("Tokens > $50:", [t["symbol"] for t in large_tokens])

10
Cheapest first: ['ARB', 'UNI', 'AAVE', 'ETH']
Most volume:    ['ETH', 'ARB', 'UNI', 'AAVE']
By price×vol:   ['ETH', 'AAVE', 'UNI', 'ARB']
ETH amounts: [1.5, 0.5, 0.25]
Tokens > $50: ['ETH', 'AAVE']


## 5. Scope — Local vs Global

**Scope** defines where a variable is accessible.
- **Local** — created inside a function, only exists there
- **Global** — created at module level, accessible everywhere

Understanding scope prevents a common class of bugs.

In [19]:
# Local scope — variable only exists inside the function
def calculate_gas_cost(gas_used, gas_price_gwei):
    gas_cost_wei = gas_used * gas_price_gwei * 10**9   # local variable
    gas_cost_eth = gas_cost_wei / 10**18               # local variable
    return gas_cost_eth

cost = calculate_gas_cost(21_000, 20)
print(f"Gas cost: {cost:.8f} ETH")

# gas_cost_wei doesn't exist out here
try:
    print(gas_cost_wei)
except NameError as e:
    print(f"NameError: {e}")   # correct — it's local to the function

Gas cost: 0.00042000 ETH
NameError: name 'gas_cost_wei' is not defined


In [20]:
# Global scope — module-level constants
ETH_DECIMALS  = 18            # module-level constant
GWEI_TO_WEI   = 10**9        # module-level constant
BLOCK_GAS_LIMIT = 30_000_000  # module-level constant

def is_gas_heavy(gas_used):
    """Check if a tx uses more than 1% of block gas limit."""
    # Reading a global is fine — no declaration needed
    return gas_used > BLOCK_GAS_LIMIT * 0.01

print(is_gas_heavy(21_000))     # False — simple transfer
print(is_gas_heavy(500_000))    # True — complex contract interaction

# Using global keyword — modify a global inside a function (use this sparingly)
request_count = 0

def fetch_price(token):
    global request_count       # declare intent to modify the global
    request_count += 1
    return {"ETH": 3247.85, "BTC": 67412.0}.get(token, 0)

fetch_price("ETH")
fetch_price("BTC")
fetch_price("UNI")
print(f"API calls made: {request_count}")   # 3

print("""
Best practice:
  - Constants at module level (ETH_DECIMALS, CHAIN_IDS, etc.)
  - Avoid global variables for state — use function parameters instead
  - Use 'global' only when absolutely necessary (e.g. a simple counter)
""")

False
True
API calls made: 3

Best practice:
  - Constants at module level (ETH_DECIMALS, CHAIN_IDS, etc.)
  - Avoid global variables for state — use function parameters instead
  - Use 'global' only when absolutely necessary (e.g. a simple counter)



## 6. Docstrings & Function Design

Good functions have docstrings — they explain what the function does,
what arguments it takes, and what it returns.
This is what appears when you call `help(your_function)`.

**Rule:** write the docstring before writing the code body.

In [21]:
def calculate_liquidation_price(
    collateral_eth: float,
    debt_usd: float,
    liquidation_threshold: float = 0.825,
    eth_price_usd: float = 3247.85,
) -> float:
    """
    Calculate the ETH price at which a position gets liquidated.

    A position is liquidated when:
        (collateral_eth * eth_price) * liquidation_threshold < debt_usd

    Args:
        collateral_eth:         ETH deposited as collateral
        debt_usd:               Total debt in USD
        liquidation_threshold:  Protocol's liquidation threshold (default: 82.5%)
        eth_price_usd:          Current ETH price in USD (default: $3,247.85)

    Returns:
        float: ETH price (USD) at which position is liquidated

    Example:
        >>> calculate_liquidation_price(10.0, 20_000.0)
        2424.24
    """
    if collateral_eth <= 0 or liquidation_threshold <= 0:
        raise ValueError("Collateral and threshold must be positive")

    liq_price = debt_usd / (collateral_eth * liquidation_threshold)
    return round(liq_price, 2)

# Use the function
collateral = 10.0    # 10 ETH
debt       = 20_000  # $20,000 USDC borrowed

liq_price  = calculate_liquidation_price(collateral, debt)
current    = 3247.85
buffer_pct = (current - liq_price) / current * 100

print(f"Collateral:          {collateral} ETH")
print(f"Debt:                ${debt:,}")
print(f"Liquidation price:   ${liq_price:,}")
print(f"Current price:       ${current:,}")
print(f"Safety buffer:       {buffer_pct:.1f}%")
print(f"Status:              {'⚠️  At risk' if buffer_pct < 20 else '✅ Safe'}")

# Built-in help uses your docstring
help(calculate_liquidation_price)

Collateral:          10.0 ETH
Debt:                $20,000
Liquidation price:   $2,424.24
Current price:       $3,247.85
Safety buffer:       25.4%
Status:              ✅ Safe
Help on function calculate_liquidation_price in module __main__:

calculate_liquidation_price(collateral_eth: float, debt_usd: float, liquidation_threshold: float = 0.825, eth_price_usd: float = 3247.85) -> float
    Calculate the ETH price at which a position gets liquidated.

    A position is liquidated when:
        (collateral_eth * eth_price) * liquidation_threshold < debt_usd

    Args:
        collateral_eth:         ETH deposited as collateral
        debt_usd:               Total debt in USD
        liquidation_threshold:  Protocol's liquidation threshold (default: 82.5%)
        eth_price_usd:          Current ETH price in USD (default: $3,247.85)

    Returns:
        float: ETH price (USD) at which position is liquidated

    Example:
        >>> calculate_liquidation_price(10.0, 20_000.0)
        24

## 7. Modules — Importing & Writing Your Own

A **module** is simply a Python file you can import and reuse.

Every `.py` file you write is a module.
When you `import` it, Python runs the file and makes its names available.

In [22]:
# Importing from the standard library
import os
import math
from datetime import datetime, timezone
from collections import defaultdict, Counter

# math — useful for blockchain calculations
print(math.log2(2**256))    # 256 — number of bits in a uint256
print(math.isclose(0.1 + 0.2, 0.3))   # True — safe float comparison

# datetime — timestamping blockchain events
now_utc = datetime.now(timezone.utc)
print(f"Current UTC: {now_utc.strftime('%Y-%m-%d %H:%M:%S')}")

# Convert a Unix timestamp (from blockchain) to readable date
block_timestamp = 1_714_000_000
dt = datetime.fromtimestamp(block_timestamp, tz=timezone.utc)
print(f"Block time: {dt.strftime('%Y-%m-%d %H:%M:%S UTC')}")

# os — useful for reading API keys from environment
api_key = os.environ.get("ETHERSCAN_API_KEY", "demo")
print(f"API key loaded: {'✅' if api_key != 'demo' else '⚠️  using demo key'}")

# Import with alias — standard conventions
import json as json_lib
import os.path as osp
print(osp.exists("/tmp"))   # True

256.0
True
Current UTC: 2026-07-08 15:01:10
Block time: 2024-04-24 23:06:40 UTC
API key loaded: ⚠️  using demo key
True


## 8. Building blockchain_utils.py

Now we put everything together into a real utility module
you'll reuse throughout this entire course.

This is what `phase-1-python-fundamentals/week-05-functions-modules/blockchain_utils.py` looks like.

In [23]:
# ── blockchain_utils.py ─────────────────────────────────────
# Your personal blockchain analytics toolkit.
# Import this in any notebook with: from blockchain_utils import *

# ── Constants ─────────────────────────────────────────────────
ETH_DECIMALS  = 18
GWEI_TO_WEI   = 10**9
WEI_PER_ETH   = 10**18

CHAIN_NAMES = {
    1:      "Ethereum",
    137:    "Polygon",
    42161:  "Arbitrum One",
    8453:   "Base",
    10:     "Optimism",
    56:     "BNB Chain",
    43114:  "Avalanche C-Chain",
    250:    "Fantom",
    100:    "Gnosis",
}

STABLECOINS = {"USDC", "USDT", "DAI", "FRAX", "LUSD", "BUSD", "TUSD", "USDP"}
MAJOR_TOKENS = {"ETH", "WETH", "BTC", "WBTC", "BNB", "SOL", "AVAX", "MATIC"}

# ── Address utilities ─────────────────────────────────────────
def shorten_address(address: str, prefix: int = 6, suffix: int = 4) -> str:
    """Shorten 0x... address to '0xABCD...1234'."""
    if len(address) <= prefix + suffix:
        return address
    return f"{address[:prefix]}...{address[-suffix:]}"

def is_valid_eth_address(address: str) -> bool:
    """Check if a string is a valid Ethereum address."""
    hex_chars = set("0123456789abcdefABCDEF")
    return (
        isinstance(address, str)
        and address.startswith("0x")
        and len(address) == 42
        and all(c in hex_chars for c in address[2:])
    )

def checksum_match(address: str) -> bool:
    """Check if address is in mixed-case EIP-55 checksum format."""
    return address != address.lower() and address != address.upper()

# ── Unit conversions ──────────────────────────────────────────
def wei_to_eth(wei: int) -> float:
    """Convert Wei (int) to ETH (float)."""
    return wei / WEI_PER_ETH

def eth_to_wei(eth: float) -> int:
    """Convert ETH (float) to Wei (int)."""
    return int(eth * WEI_PER_ETH)

def gwei_to_eth(gwei: float) -> float:
    """Convert Gwei to ETH."""
    return gwei / 1_000_000_000

def token_amount(raw: int, decimals: int = 18) -> float:
    """Convert raw token amount to human-readable float."""
    return raw / 10**decimals

def format_token(raw: int, symbol: str, decimals: int = 18, precision: int = 6) -> str:
    """Format a raw token amount with symbol."""
    return f"{token_amount(raw, decimals):,.{precision}f} {symbol}"

# ── Gas utilities ─────────────────────────────────────────────
def gas_cost_eth(gas_used: int, gas_price_gwei: float) -> float:
    """Calculate gas cost in ETH."""
    return gas_used * gas_price_gwei * GWEI_TO_WEI / WEI_PER_ETH

def gas_cost_usd(gas_used: int, gas_price_gwei: float, eth_price_usd: float) -> float:
    """Calculate gas cost in USD."""
    return gas_cost_eth(gas_used, gas_price_gwei) * eth_price_usd

def gas_tier(gwei: float) -> str:
    """Return a human-readable gas price tier."""
    if gwei < 10:   return "🟢 Low"
    elif gwei < 30: return "🟡 Normal"
    elif gwei < 50: return "🟠 High"
    elif gwei < 100:return "🔴 Very High"
    else:           return "🚨 Extreme"

# ── Price utilities ───────────────────────────────────────────
def eth_to_usd(eth: float, eth_price_usd: float) -> float:
    """Convert ETH amount to USD."""
    return eth * eth_price_usd

def pnl(cost_basis_usd: float, current_value_usd: float) -> tuple:
    """Return (pnl_usd, pnl_pct) for a position."""
    pnl_usd = current_value_usd - cost_basis_usd
    pnl_pct = (pnl_usd / cost_basis_usd * 100) if cost_basis_usd else 0
    return round(pnl_usd, 2), round(pnl_pct, 2)

def price_impact(amount_usd: float, liquidity_usd: float) -> float:
    """Estimate price impact of a trade given pool liquidity."""
    return (amount_usd / liquidity_usd) * 100

# ── Token classification ──────────────────────────────────────
def token_category(symbol: str) -> str:
    """Classify a token symbol into a category."""
    if symbol in STABLECOINS:   return "stablecoin"
    if symbol in MAJOR_TOKENS:  return "major"
    return "altcoin"

def is_stablecoin(symbol: str) -> bool:
    """Return True if the symbol is a known stablecoin."""
    return symbol in STABLECOINS

# ── Wallet classification ─────────────────────────────────────
def classify_wallet(tx_count: int, balance_eth: float,
                    has_contract_interactions: bool = False) -> str:
    """
    Classify a wallet based on activity metrics.

    Args:
        tx_count:                    Total transaction count
        balance_eth:                 Current ETH balance
        has_contract_interactions:   Whether wallet interacts with contracts

    Returns:
        str: Wallet label with emoji
    """
    if tx_count > 10_000:                    return "🤖 Bot / High-frequency"
    if balance_eth > 1_000:                  return "🐋 Whale"
    if balance_eth > 10 and tx_count > 100:  return "🐬 Dolphin"
    if has_contract_interactions:            return "⚡ DeFi User"
    if tx_count > 50:                        return "👤 Active Retail"
    return "🐣 New / Inactive"

# ── Chain utilities ───────────────────────────────────────────
def chain_name(chain_id: int) -> str:
    """Return human-readable chain name for a chain ID."""
    return CHAIN_NAMES.get(chain_id, f"Unknown Chain ({chain_id})")

def get_explorer_url(chain_id: int, tx_hash: str) -> str:
    """Return block explorer URL for a transaction."""
    explorers = {
        1:      "https://etherscan.io/tx",
        137:    "https://polygonscan.com/tx",
        42161:  "https://arbiscan.io/tx",
        8453:   "https://basescan.org/tx",
        10:     "https://optimistic.etherscan.io/tx",
    }
    base = explorers.get(chain_id, "https://etherscan.io/tx")
    return f"{base}/{tx_hash}"

# ── Formatting helpers ────────────────────────────────────────
def fmt_usd(value: float, decimals: int = 2) -> str:
    """Format a USD value with $ prefix and comma separation."""
    return f"${value:,.{decimals}f}"

def fmt_eth(value: float, decimals: int = 4) -> str:
    """Format an ETH value."""
    return f"{value:,.{decimals}f} ETH"

def fmt_pct(value: float, decimals: int = 2, sign: bool = True) -> str:
    """Format a percentage value."""
    fmt = f"{'+' if sign else ''}{value:.{decimals}f}%"
    return fmt

def fmt_large(value: float) -> str:
    """Format large numbers as K, M, B."""
    if abs(value) >= 1e9:   return f"${value/1e9:.2f}B"
    if abs(value) >= 1e6:   return f"${value/1e6:.2f}M"
    if abs(value) >= 1e3:   return f"${value/1e3:.2f}K"
    return f"${value:.2f}"

# ── Demo ─────────────────────────────────────────────────────
if __name__ == "__main__":
    print("blockchain_utils.py — self test")
    print("-" * 40)

    addr = "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045"
    print(f"Address:       {shorten_address(addr)}")
    print(f"Valid:         {is_valid_eth_address(addr)}")
    print(f"1.5 ETH wei:   {eth_to_wei(1.5):,}")
    print(f"1.5e18 wei:    {wei_to_eth(1_500_000_000_000_000_000)} ETH")
    print(f"Gas cost:      {gas_cost_usd(21_000, 20, 3247.85):.4f} USD")
    print(f"Gas tier:      {gas_tier(35)}")
    print(f"USDC type:     {token_category('USDC')}")
    print(f"UNI type:      {token_category('UNI')}")
    print(f"Wallet class:  {classify_wallet(250, 15.0, True)}")
    print(f"Chain 42161:   {chain_name(42161)}")
    print(f"Explorer URL:  {get_explorer_url(1, '0xabc123')}")
    pnl_usd, pnl_pct = pnl(10_000, 12_500)
    print(f"P&L:           {fmt_usd(pnl_usd)} ({fmt_pct(pnl_pct)})")
    print(f"Large fmt:     {fmt_large(5_800_000_000)}")
    print("-" * 40)
    print("All checks passed ✓")

blockchain_utils.py — self test
----------------------------------------
Address:       0xd8dA...6045
Valid:         True
1.5 ETH wei:   1,500,000,000,000,000,000
1.5e18 wei:    1.5 ETH
Gas cost:      1.3641 USD
Gas tier:      🟠 High
USDC type:     stablecoin
UNI type:      altcoin
Wallet class:  🐬 Dolphin
Chain 42161:   Arbitrum One
Explorer URL:  https://etherscan.io/tx/0xabc123
P&L:           $2,500.00 (+25.00%)
Large fmt:     $5.80B
----------------------------------------
All checks passed ✓


## Summary

| Concept | Syntax | Blockchain use |
|---------|--------|----------------|
| `def` | `def fn(param):` | Reusable logic for addresses, prices, gas |
| Default args | `def fn(x, y=10):` | Optional API params, precision settings |
| Keyword args | `fn(x=1, y=2)` | Explicit, readable function calls |
| `*args` | `def fn(*vals):` | Batch address/value processing |
| `**kwargs` | `def fn(**opts):` | Flexible API request builders |
| `lambda` | `lambda x: x*2` | Key functions for sorting tokens |
| Scope | local / global | Constants at module level, logic local |
| Modules | `import blockchain_utils` | Shared utility toolkit |
| Docstrings | `""" ... """` | Self-documenting analytics functions |

---

## What's next

**Week 6 — OOP:** Take everything you've built in functions and wrap it into
classes — a `Wallet` class, a `Token` class, a `Transaction` class.
This is the last week of Phase 1 before moving into data analysis.

---

**Your task before Week 6:**
1. Complete `exercises.py`
2. Copy `blockchain_utils.py` into your repo — you'll import it from Week 6 onwards
3. Add at least 2 of your own utility functions to `blockchain_utils.py`
4. Commit:
```bash
git add phase-1-python-fundamentals/week-05-functions-modules/
git commit -m "phase-1/week-05: functions and modules — lesson + exercises"
git push
```